# Modelagem

Dois modelos: Regressão Logística e Random Forest. Validação pra comparar, teste fechado pro resultado final.

## 1. Imports e carregamento

In [1]:
import copy
from matplotlib.path import Path as _MplPath
def _fixed_path_deepcopy(self, memo):
    cls = self.__class__
    p = cls.__new__(cls)
    memo[id(self)] = p
    for k, v in self.__dict__.items():
        try:
            setattr(p, k, copy.deepcopy(v, memo))
        except Exception:
            setattr(p, k, v)
    p._readonly = False
    return p
_MplPath.__deepcopy__ = _fixed_path_deepcopy

import matplotlib
matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, precision_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

os.makedirs('figures', exist_ok=True)
os.makedirs('models', exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

In [2]:
train = pd.read_csv('data/processed/train.csv')
val   = pd.read_csv('data/processed/val.csv')
test  = pd.read_csv('data/processed/test.csv')

X_train, y_train = train.drop(columns='diagnosis'), train['diagnosis']
X_val,   y_val   = val.drop(columns='diagnosis'),   val['diagnosis']
X_test,  y_test  = test.drop(columns='diagnosis'),  test['diagnosis']

print(f'Treino:    {X_train.shape[0]} amostras | Malignos: {y_train.sum()} ({y_train.mean():.1%})')
print(f'Validação: {X_val.shape[0]} amostras  | Malignos: {y_val.sum()} ({y_val.mean():.1%})')
print(f'Teste:     {X_test.shape[0]} amostras  | Malignos: {y_test.sum()} ({y_test.mean():.1%})')

Treino:    386 amostras | Malignos: 144 (37.3%)
Validação: 69 amostras  | Malignos: 26 (37.7%)
Teste:     114 amostras  | Malignos: 42 (36.8%)


## 2. A métrica que importa

Com 63% benignos, accuracy sozinha mente. E num contexto médico, falso negativo (maligno classificado como benigno) é muito pior que falso positivo. O critério de escolha é **recall da classe M**.

## 3. Regressão Logística

In [3]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_val_pred_lr = lr.predict(X_val)

print('=== Regressão Logística — Validação ===')
print(classification_report(y_val, y_val_pred_lr, target_names=['Benigno', 'Maligno']))

=== Regressão Logística — Validação ===
              precision    recall  f1-score   support

     Benigno       0.98      0.98      0.98        43
     Maligno       0.96      0.96      0.96        26

    accuracy                           0.97        69
   macro avg       0.97      0.97      0.97        69
weighted avg       0.97      0.97      0.97        69



/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_po

## 4. Random Forest

In [4]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_val_pred_rf = rf.predict(X_val)

print('=== Random Forest — Validação ===')
print(classification_report(y_val, y_val_pred_rf, target_names=['Benigno', 'Maligno']))

=== Random Forest — Validação ===
              precision    recall  f1-score   support

     Benigno       0.96      1.00      0.98        43
     Maligno       1.00      0.92      0.96        26

    accuracy                           0.97        69
   macro avg       0.98      0.96      0.97        69
weighted avg       0.97      0.97      0.97        69



## 5. Comparação na validação

In [5]:
def metrics(y_true, y_pred, model_name):
    return {
        'Modelo':     model_name,
        'Accuracy':   accuracy_score(y_true, y_pred),
        'Recall (M)': recall_score(y_true, y_pred),
        'Precision':  precision_score(y_true, y_pred),
        'F1 (M)':     f1_score(y_true, y_pred),
    }

results_val = pd.DataFrame([
    metrics(y_val, y_val_pred_lr, 'Logistic Regression'),
    metrics(y_val, y_val_pred_rf, 'Random Forest'),
]).set_index('Modelo').round(4)

print('=== Validação — Comparação ===')
print(results_val.to_string())

=== Validação — Comparação ===
                     Accuracy  Recall (M)  Precision  F1 (M)
Modelo                                                      
Logistic Regression     0.971      0.9615     0.9615  0.9615
Random Forest           0.971      0.9231     1.0000  0.9600


## 6. Matrizes de confusão — validação

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(
    axes,
    [y_val_pred_lr, y_val_pred_rf],
    ['Regressão Logística', 'Random Forest']
):
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Benigno', 'Maligno'],
                yticklabels=['Benigno', 'Maligno'])
    ax.set_title(title)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predito')

plt.suptitle('Matrizes de Confusão — Validação', fontsize=13, fontweight='bold')
plt.savefig('figures/07_confusion_matrices_val.png', bbox_inches='tight')
plt.show()
print('[OK]')

[OK]


/var/folders/c0/pbls_tj56jn3kq8p6gckzcc40000gn/T/ipykernel_30942/2904931142.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Curvas ROC — validação

In [7]:
fig, ax = plt.subplots(figsize=(7, 6))

for model, name, color in [
    (lr, 'Logistic Regression', '#2c7fb8'),
    (rf, 'Random Forest',       '#fc8d62'),
]:
    proba = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc = roc_auc_score(y_val, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('Taxa de Falso Positivo')
ax.set_ylabel('Taxa de Verdadeiro Positivo (Recall)')
ax.set_title('Curva ROC — Validação')
ax.legend()
plt.savefig('figures/08_roc_curves_val.png', bbox_inches='tight')
plt.show()
print('[OK]')

[OK]


/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/c0/pbls_tj56jn3kq8p6gckzcc40000gn/T/ipykernel_30942/791043292.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Qual modelo escolher?

Accuracy idêntica (97%). O que separa: RF teve recall 92.3% (2 falsos negativos), LR teve 96.1% (1 falso negativo). Pelo critério de recall de M, a **Regressão Logística** fica.

In [8]:
recall_lr = recall_score(y_val, y_val_pred_lr)
recall_rf = recall_score(y_val, y_val_pred_rf)

print(f'Recall (M) — Logistic Regression: {recall_lr:.4f}')
print(f'Recall (M) — Random Forest:       {recall_rf:.4f}')
print()

best_model = lr if recall_lr >= recall_rf else rf
best_name  = 'Logistic Regression' if recall_lr >= recall_rf else 'Random Forest'
print(f'Modelo selecionado: {best_name}')

Recall (M) — Logistic Regression: 0.9615
Recall (M) — Random Forest:       0.9231

Modelo selecionado: Logistic Regression


## 9. Avaliação no teste

Conjunto de teste abrindo pela primeira vez. Esse é o número final.

In [9]:
y_test_pred = best_model.predict(X_test)

print(f'=== {best_name} — TESTE FINAL ===\n')
print(classification_report(y_test, y_test_pred, target_names=['Benigno', 'Maligno']))

results_test = metrics(y_test, y_test_pred, best_name)
for k, v in results_test.items():
    if k != 'Modelo':
        print(f'  {k}: {v:.4f}')

=== Logistic Regression — TESTE FINAL ===

              precision    recall  f1-score   support

     Benigno       0.97      0.99      0.98        72
     Maligno       0.98      0.95      0.96        42

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114

  Accuracy: 0.9737
  Recall (M): 0.9524
  Precision: 0.9756
  F1 (M): 0.9639


/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [10]:
# Ambos os modelos no teste, pra registro completo
y_test_pred_lr = lr.predict(X_test)
y_test_pred_rf = rf.predict(X_test)

results_test_all = pd.DataFrame([
    metrics(y_test, y_test_pred_lr, 'Logistic Regression'),
    metrics(y_test, y_test_pred_rf, 'Random Forest'),
]).set_index('Modelo').round(4)

print('=== Teste Final — Ambos os Modelos ===')
print(results_test_all.to_string())

=== Teste Final — Ambos os Modelos ===
                     Accuracy  Recall (M)  Precision  F1 (M)
Modelo                                                      
Logistic Regression    0.9737      0.9524     0.9756  0.9639
Random Forest          0.9737      0.9286     1.0000  0.9630


/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/annamaria/src/fiap/fase1/B/.venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 10. Matriz de confusão — teste final

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(
    axes,
    [y_test_pred_lr, y_test_pred_rf],
    ['Regressão Logística', 'Random Forest']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax,
                xticklabels=['Benigno', 'Maligno'],
                yticklabels=['Benigno', 'Maligno'])
    ax.set_title(f'{title} (Teste)')
    ax.set_ylabel('Real')
    ax.set_xlabel('Predito')

plt.suptitle('Matrizes de Confusão — Teste Final', fontsize=13, fontweight='bold')
plt.savefig('figures/09_confusion_matrices_test.png', bbox_inches='tight')
plt.show()
print('[OK]')

[OK]


/var/folders/c0/pbls_tj56jn3kq8p6gckzcc40000gn/T/ipykernel_30942/4264052308.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Salva os modelos

In [12]:
joblib.dump(lr, 'models/logistic_regression.pkl')
joblib.dump(rf, 'models/random_forest.pkl')
joblib.dump(best_model, 'models/best_model.pkl')

print('Modelos salvos:')
for f in os.listdir('models'):
    path = f'models/{f}'
    print(f'  {path} ({os.path.getsize(path)/1024:.1f} KB)')

Modelos salvos:
  models/best_model.pkl (1.8 KB)
  models/logistic_regression.pkl (1.8 KB)
  models/preprocessing_pipeline.pkl (2.1 KB)
  models/random_forest.pkl (582.2 KB)


97.4% de accuracy nos dois. O RF não teve falso positivo (precision 100%), mas deixou passar 3 malignos. A LR deixou 2. O modelo não substitui o médico; serve como suporte na triagem.

In [13]:
print('=== Resumo de resultados ===')
print(f'\nModelo selecionado: {best_name}')
print('\nValidação:')
print(results_val.to_string())
print('\nTeste:')
print(results_test_all.to_string())

# Falsos negativos no teste (malignos classificados como benignos)
for y_pred, name in [(y_test_pred_lr, 'LR'), (y_test_pred_rf, 'RF')]:
    fn = ((y_test == 1) & (pd.Series(y_pred) == 0)).sum()
    total_m = (y_test == 1).sum()
    print(f'\nFalsos negativos ({name}): {fn} de {total_m} malignos ({fn/total_m:.1%})')

=== Resumo de resultados ===

Modelo selecionado: Logistic Regression

Validação:
                     Accuracy  Recall (M)  Precision  F1 (M)
Modelo                                                      
Logistic Regression     0.971      0.9615     0.9615  0.9615
Random Forest           0.971      0.9231     1.0000  0.9600

Teste:
                     Accuracy  Recall (M)  Precision  F1 (M)
Modelo                                                      
Logistic Regression    0.9737      0.9524     0.9756  0.9639
Random Forest          0.9737      0.9286     1.0000  0.9630

Falsos negativos (LR): 2 de 42 malignos (4.8%)

Falsos negativos (RF): 3 de 42 malignos (7.1%)
